In [5]:
import pandas as pd
import re
import ast
import numpy as np
import gensim.downloader as api
import torch
import torch.nn as nn
import torch.optim as optim
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import classification_report
from torch.utils.data import DataLoader, TensorDataset
from imblearn.over_sampling import RandomOverSampler 

# 1. Cargar el conjunto de datos desde una ruta absoluta
df = pd.read_csv('tmdb_5000_movies.csv')

df = df[['overview', 'genres']]
df['main_genre'] = df['genres'].apply(lambda x: ast.literal_eval(x)[0]['name'] if x != '[]' else None)
df = df.dropna(subset=['overview', 'main_genre'])

# 2. Preprocesamiento del texto
def preprocess_text(text):
    text = text.lower()
    text = re.sub(r'\W', ' ', text)
    text = re.sub(r'\s+', ' ', text)
    tokens = text.split()
    stopwords = set(["the", "is", "and", "in", "it", "to", "for", "a", "of", "with"])  # Stopwords personalizadas
    tokens = [word for word in tokens if word not in stopwords]
    return ' '.join(tokens)

df['processed_overview'] = df['overview'].apply(preprocess_text)

# 3. Obtener embeddings con GloVe
model = api.load("glove-wiki-gigaword-100")

def get_embedding(text, model):
    tokens = text.split()
    embeddings = [model[word] for word in tokens if word in model]
    return np.mean(embeddings, axis=0) if embeddings else np.zeros(100)

df['embedding'] = df['processed_overview'].apply(lambda x: get_embedding(x, model))

# 4. Convertir géneros en valores numéricos
le = LabelEncoder()
df['genre_encoded'] = le.fit_transform(df['main_genre'])

# 5. Crear conjuntos de entrenamiento y validación
X = np.vstack(df['embedding'].values)
y = df['genre_encoded'].values

# Aplicar sobremuestreo para lidiar con el desequilibrio de clases
ros = RandomOverSampler(random_state=42)
X_resampled, y_resampled = ros.fit_resample(X, y)

# Dividir los datos
X_train, X_val, y_train, y_val = train_test_split(X_resampled, y_resampled, test_size=0.2, random_state=42)

# 6. Convertir a tensores y crear DataLoader para lotes
train_dataset = TensorDataset(torch.tensor(X_train, dtype=torch.float32), torch.tensor(y_train, dtype=torch.long))
val_dataset = TensorDataset(torch.tensor(X_val, dtype=torch.float32), torch.tensor(y_val, dtype=torch.long))
train_loader = DataLoader(train_dataset, batch_size=64, shuffle=True, num_workers=4)
val_loader = DataLoader(val_dataset, batch_size=64, shuffle=False, num_workers=4)

# 7. Definir el modelo mejorado con LSTM
class ImprovedGenreClassifier(nn.Module):
    def __init__(self, input_size, hidden_size, output_size):
        super(ImprovedGenreClassifier, self).__init__()
        self.lstm = nn.LSTM(input_size, hidden_size, batch_first=True)
        self.fc1 = nn.Linear(hidden_size, hidden_size // 2)
        self.fc2 = nn.Linear(hidden_size // 2, output_size)
        self.dropout = nn.Dropout(0.5)
        self.relu = nn.ReLU()

    def forward(self, x):
        x = x.unsqueeze(1)  # Añadir dimensión para LSTM
        lstm_out, _ = self.lstm(x)
        out = lstm_out[:, -1, :]  # Usar la última salida del LSTM
        out = self.fc1(out)
        out = self.relu(out)
        out = self.dropout(out)
        out = self.fc2(out)
        return out

# Inicializar el modelo, función de pérdida y optimizador
model = ImprovedGenreClassifier(input_size=100, hidden_size=128, output_size=len(le.classes_))

# Pesos de las clases para manejar el desequilibrio de clases
class_weights = torch.tensor([1.0 / np.bincount(y_train)[i] for i in range(len(le.classes_))], dtype=torch.float32)

criterion = nn.CrossEntropyLoss(weight=class_weights)  # Función de pérdida ponderada
optimizer = optim.Adam(model.parameters(), lr=0.001, weight_decay=1e-5) # Optimizador Adam

# 8. Función de entrenamiento mejorada con evaluación
def train_model(model, train_loader, val_loader, criterion, optimizer, epochs=20, report_interval=5):
    for epoch in range(epochs):
        model.train()
        running_loss = 0.0
        for inputs, labels in train_loader:
            optimizer.zero_grad()
            outputs = model(inputs)
            loss = criterion(outputs, labels)
            loss.backward()
            optimizer.step()
            running_loss += loss.item()

        # Mostrar resultados cada `report_interval` épocas
        if (epoch + 1) % report_interval == 0:
            model.eval()
            val_loss = 0.0
            all_preds, all_labels = [], []
            with torch.no_grad():
                for inputs, labels in val_loader:
                    outputs = model(inputs)
                    loss = criterion(outputs, labels)
                    val_loss += loss.item()
                    all_preds.append(outputs.argmax(dim=1))
                    all_labels.append(labels)
            
            all_preds = torch.cat(all_preds)
            all_labels = torch.cat(all_labels)

            print(f'Época {epoch+1}, Pérdida entrenamiento: {running_loss / len(train_loader)}')
            print(f'Época {epoch+1}, Pérdida validación: {val_loss / len(val_loader)}')

            unique_labels = torch.unique(all_labels, dim=0)
            print(classification_report(all_labels, all_preds, labels=unique_labels, target_names=[le.classes_[i] for i in unique_labels]))

# 9. Entrenar el modelo
train_model(model, train_loader, val_loader, criterion, optimizer, epochs=20, report_interval=5)

Época 5, Pérdida entrenamiento: 1.630253651678957
Época 5, Pérdida validación: 1.459736356609746
                 precision    recall  f1-score   support

         Action       0.24      0.15      0.18       248
      Adventure       0.26      0.05      0.09       224
      Animation       0.37      0.41      0.39       232
         Comedy       0.28      0.18      0.22       262
          Crime       0.38      0.39      0.38       257
    Documentary       0.63      0.78      0.70       239
          Drama       0.25      0.08      0.12       238
         Family       0.39      0.55      0.46       238
        Fantasy       0.28      0.27      0.28       223
        Foreign       0.98      1.00      0.99       248
        History       0.78      0.96      0.86       239
         Horror       0.49      0.25      0.33       245
          Music       0.65      0.92      0.76       235
        Mystery       0.42      0.66      0.52       244
        Romance       0.21      0.29      0.24 